# Question 6 - Writing More Tests

We expand coverage across five SQL-agent scenarios. Ground truth comes directly from DuckDB, independently of the model.

```text
Scenario
    ↓
known DuckDB ground truth
    ↓
run SQL agent
    ↓
inspect generated SQL
    ↓
inspect tool trajectory
    ↓
compare result
    ↓
PASS / FAIL
```

The tests check schema columns, schema-first tool calls, and result values. Integer values must match exactly; averages use `math.isclose` with `1e-6` relative/absolute tolerance. A small shared assertion helper in `test_agent.py` handles these repeated checks without another LLM judge.

Credit-card payments use `payment_type = 1`, as documented in the [TLC data dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf). DuckDB's `DOW` is verified below: Sunday=0, Wednesday=3. Day names are also valid agent answers.

Run from `week3_homework/` with the project dependencies and `OPENAI_API_KEY` available.

In [1]:
from sql_agent import agent
from sql_tools import con
from test_agent import _assert_scenario

schema = con.execute("DESCRIBE trips").fetchall()
assert any(row[0] == "passenger_count" for row in schema)
print("Verified passenger column:", next(row[:2] for row in schema if row[0] == "passenger_count"))
print("Sunday DOW check:", con.execute("SELECT EXTRACT(DOW FROM DATE '2024-01-07'), dayname(DATE '2024-01-07')").fetchone())

Verified passenger column: ('passenger_count', 'BIGINT')
Sunday DOW check: (0, 'Sunday')


### Average tip for credit-card payments

What is the average tip amount for credit card payments?

In [2]:
verification_query = 'SELECT AVG(tip_amount) FROM trips WHERE payment_type = 1;'
expected_row = con.execute(verification_query).fetchone()
print("Independent SQL:", verification_query)
print("DuckDB answer:", expected_row)

Independent SQL: SELECT AVG(tip_amount) FROM trips WHERE payment_type = 1;
DuckDB answer: (4.1696706274903645,)


In [3]:
user_prompt = 'What is the average tip amount for credit card payments?'
print("Prompt:", user_prompt)
result = await agent.run(user_prompt)
numbers = _assert_scenario(result, ['tip_amount', 'payment_type'], expected_row)
print("Verified columns:", ['tip_amount', 'payment_type'])
print("PASS: schema-first trajectory, required columns, and known result values.")

Prompt: What is the average tip amount for credit card payments?


                 pydantic-ai v2.63.0 • Python 3.13.15

      / \        agent: agent • model: openai:gpt-4o-mini • output: SQLResult • tools: 2
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


SQLResult: {
  "sql_query": "SELECT AVG(tip_amount) AS average_tip_amount \nFROM trips \nWHERE payment_type = 1;",
  "result_text": "average_tip_amount\n4.169670627490263",
  "row_count": 1
}
Tool calls: [ToolCall(name='get_schema', args={}), ToolCall(name='run_sql', args={'query': 'SELECT AVG(tip_amount) AS average_tip_amount \nFROM trips \nWHERE payment_type = 1;'})]
Verified columns: ['tip_amount', 'payment_type']
PASS: schema-first trajectory, required columns, and known result values.


### Pickup location with the most trips

Which pickup location (PULocationID) has the most trips?

In [4]:
verification_query = 'SELECT PULocationID, COUNT(*) AS trip_count FROM trips GROUP BY PULocationID ORDER BY trip_count DESC LIMIT 1;'
expected_row = con.execute(verification_query).fetchone()
print("Independent SQL:", verification_query)
print("DuckDB answer:", expected_row)

Independent SQL: SELECT PULocationID, COUNT(*) AS trip_count FROM trips GROUP BY PULocationID ORDER BY trip_count DESC LIMIT 1;
DuckDB answer: (132, 145240)


In [5]:
user_prompt = 'Which pickup location (PULocationID) has the most trips?'
print("Prompt:", user_prompt)
result = await agent.run(user_prompt)
numbers = _assert_scenario(result, ['PULocationID'], expected_row)
print("Verified columns:", ['PULocationID'])
print("PASS: schema-first trajectory, required columns, and known result values.")

Prompt: Which pickup location (PULocationID) has the most trips?


SQLResult: {
  "sql_query": "SELECT PULocationID, COUNT(*) AS trip_count\nFROM trips\nGROUP BY PULocationID\nORDER BY trip_count DESC\nLIMIT 1;",
  "result_text": "PULocationID\ttrip_count\n132\t145240",
  "row_count": 1
}
Tool calls: [ToolCall(name='get_schema', args={}), ToolCall(name='run_sql', args={'query': 'SELECT PULocationID, COUNT(*) AS trip_count\nFROM trips\nGROUP BY PULocationID\nORDER BY trip_count DESC\nLIMIT 1;'})]
Verified columns: ['PULocationID']
PASS: schema-first trajectory, required columns, and known result values.


### Average fare for trips longer than 10 miles

What is the average fare for trips longer than 10 miles?

In [6]:
verification_query = 'SELECT AVG(fare_amount) FROM trips WHERE trip_distance > 10;'
expected_row = con.execute(verification_query).fetchone()
print("Independent SQL:", verification_query)
print("DuckDB answer:", expected_row)

Independent SQL: SELECT AVG(fare_amount) FROM trips WHERE trip_distance > 10;
DuckDB answer: (62.8808124282597,)


In [7]:
user_prompt = 'What is the average fare for trips longer than 10 miles?'
print("Prompt:", user_prompt)
result = await agent.run(user_prompt)
numbers = _assert_scenario(result, ['fare_amount', 'trip_distance'], expected_row)
print("Verified columns:", ['fare_amount', 'trip_distance'])
print("PASS: schema-first trajectory, required columns, and known result values.")

Prompt: What is the average fare for trips longer than 10 miles?


SQLResult: {
  "sql_query": "SELECT AVG(fare_amount) AS average_fare\nFROM trips\nWHERE trip_distance > 10;",
  "result_text": "62.88081242825971",
  "row_count": 1
}
Tool calls: [ToolCall(name='get_schema', args={}), ToolCall(name='run_sql', args={'query': 'SELECT AVG(fare_amount) AS average_fare\nFROM trips\nWHERE trip_distance > 10;'})]
Verified columns: ['fare_amount', 'trip_distance']
PASS: schema-first trajectory, required columns, and known result values.


### Trips with zero passengers

How many trips had zero passengers recorded?

In [8]:
verification_query = 'SELECT COUNT(*) FROM trips WHERE passenger_count = 0;'
expected_row = con.execute(verification_query).fetchone()
print("Independent SQL:", verification_query)
print("DuckDB answer:", expected_row)

Independent SQL: SELECT COUNT(*) FROM trips WHERE passenger_count = 0;
DuckDB answer: (31465,)


In [9]:
user_prompt = 'How many trips had zero passengers recorded?'
print("Prompt:", user_prompt)
result = await agent.run(user_prompt)
numbers = _assert_scenario(result, ['passenger_count'], expected_row)
print("Verified columns:", ['passenger_count'])
print("PASS: schema-first trajectory, required columns, and known result values.")

Prompt: How many trips had zero passengers recorded?


SQLResult: {
  "sql_query": "SELECT COUNT(*) AS zero_passenger_count FROM trips WHERE passenger_count = 0;",
  "result_text": "zero_passenger_count\n31465",
  "row_count": 1
}
Tool calls: [ToolCall(name='get_schema', args={}), ToolCall(name='run_sql', args={'query': 'SELECT COUNT(*) AS zero_passenger_count FROM trips WHERE passenger_count = 0;'})]
Verified columns: ['passenger_count']
PASS: schema-first trajectory, required columns, and known result values.


### Busiest day of the week

What is the busiest day of the week for taxi trips?

In [10]:
verification_query = 'SELECT EXTRACT(DOW FROM tpep_pickup_datetime) AS dow, dayname(tpep_pickup_datetime) AS weekday, COUNT(*) AS trip_count FROM trips GROUP BY dow, weekday ORDER BY trip_count DESC LIMIT 1;'
expected_row = con.execute(verification_query).fetchone()
print("Independent SQL:", verification_query)
print("DuckDB answer:", expected_row)

Independent SQL: SELECT EXTRACT(DOW FROM tpep_pickup_datetime) AS dow, dayname(tpep_pickup_datetime) AS weekday, COUNT(*) AS trip_count FROM trips GROUP BY dow, weekday ORDER BY trip_count DESC LIMIT 1;
DuckDB answer: (3, 'Wednesday', 495032)


In [11]:
user_prompt = 'What is the busiest day of the week for taxi trips?'
print("Prompt:", user_prompt)
result = await agent.run(user_prompt)
numbers = _assert_scenario(result, ['tpep_pickup_datetime'], [expected_row[2]])
assert expected_row[1].lower() in result.output.result_text.lower() or expected_row[0] in numbers
print("Verified columns:", ['tpep_pickup_datetime'])
print("PASS: schema-first trajectory, required columns, and known result values.")

Prompt: What is the busiest day of the week for taxi trips?


SQLResult: {
  "sql_query": "SELECT strftime('%A', tpep_pickup_datetime) AS day_of_week, COUNT(*) AS trip_count\nFROM trips\nGROUP BY day_of_week\nORDER BY trip_count DESC\nLIMIT 1;",
  "result_text": "day_of_week\ttrip_count\nWednesday\t495032",
  "row_count": 1
}
Tool calls: [ToolCall(name='get_schema', args={}), ToolCall(name='run_sql', args={'query': "SELECT strftime('%A', tpep_pickup_datetime) AS day_of_week, COUNT(*) AS trip_count\nFROM trips\nGROUP BY day_of_week\nORDER BY trip_count DESC\nLIMIT 1;"})]
Verified columns: ['tpep_pickup_datetime']
PASS: schema-first trajectory, required columns, and known result values.


For **"How many trips had zero passengers recorded?"**, the SQL query should filter on **`passenger_count`**, with `passenger_count = 0`.

This column is present in the actual schema. The zero-passenger scenario above displays the real generated SQL and checks the independently known count.